# Beyond Basic RAG: Hybrid Search Implementation

This notebook demonstrates the implementation of hybrid search combining BM25 sparse retrieval with dense vector search.

## Table of Contents
1. Setup and Installation
2. Load Sample Documents
3. BM25 Sparse Retrieval
4. Dense Vector Retrieval
5. Hybrid Search with RRF
6. Cross-Encoder Reranking
7. Context Optimization
8. Complete Pipeline

## 1. Setup and Installation

In [ ]:
# Install required packages (uncomment if needed)
# !pip install -r ../requirements.txt

import sys
sys.path.append('..')

from src.retrieval import HybridRetriever, create_sample_documents
from src.reranker import CrossEncoderReranker, RerankedRetriever
from src.utils import reorder_documents_lost_in_middle, calculate_context_stats

print("✓ Imports successful")

## 2. Load Sample Documents

In [ ]:
# Load sample knowledge base
documents = create_sample_documents()

print(f"Loaded {len(documents)} documents\n")
for i, doc in enumerate(documents, 1):
    print(f"{i}. [{doc.metadata['doc_id']}] {doc.page_content[:80]}...")

## 3. Test BM25 Sparse Retrieval

In [ ]:
from langchain_community.retrievers import BM25Retriever

# Initialize BM25 retriever
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 3

# Test exact keyword match
query = "503-AUTH-TIMEOUT"
print(f"Query: {query}\n")

results = bm25_retriever.get_relevant_documents(query)
print("BM25 Results:")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.metadata['doc_id']}: {doc.page_content[:100]}...")

## 4. Test Dense Vector Retrieval

In [ ]:
# Note: Requires OPENAI_API_KEY environment variable
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(documents, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Test semantic query
query = "How do I fix authentication timeouts?"
print(f"Query: {query}\n")

results = dense_retriever.get_relevant_documents(query)
print("Dense Vector Results:")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.metadata['doc_id']}: {doc.page_content[:100]}...")

## 5. Hybrid Search with Reciprocal Rank Fusion

In [ ]:
# Initialize hybrid retriever
hybrid_retriever = HybridRetriever(
    documents=documents,
    bm25_weight=0.5,
    dense_weight=0.5
)

# Test hybrid search
query = "How do I fix authentication timeouts?"
print(f"Query: {query}\n")

results = hybrid_retriever.get_relevant_documents(query, top_k=5)
print("Hybrid Search Results (RRF):")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.metadata['doc_id']}: {doc.page_content[:100]}...")

## 6. Cross-Encoder Reranking

In [ ]:
# Initialize reranker
reranker = CrossEncoderReranker(model_name="fast")

# Get candidates from hybrid search
candidates = hybrid_retriever.get_relevant_documents(query, top_k=6)

# Rerank with cross-encoder
reranked = reranker.rerank(query, candidates, top_k=3)

print(f"Query: {query}\n")
print("Reranked Results:")
for i, doc in enumerate(reranked, 1):
    score = doc.metadata.get('rerank_score', 0)
    print(f"{i}. [Score: {score:.3f}] {doc.metadata['doc_id']}")
    print(f"   {doc.page_content[:120]}...\n")

## 7. Context Optimization (Lost in the Middle Fix)

In [ ]:
# Demonstrate document reordering
print("Original order (by relevance score):")
for i, doc in enumerate(reranked, 1):
    print(f"{i}. {doc.metadata['doc_id']}")

# Apply reordering
optimized = reorder_documents_lost_in_middle(reranked)

print("\nReordered (Lost in the Middle optimization):")
for i, doc in enumerate(optimized, 1):
    print(f"{i}. {doc.metadata['doc_id']}")

print("\n✓ Most relevant docs are now at START and END positions")

## 8. Complete Production Pipeline

In [ ]:
# Create production-ready retriever
production_retriever = RerankedRetriever(
    base_retriever=hybrid_retriever,
    reranker=reranker,
    top_k=5,
    apply_context_reordering=True
)

# Test queries
test_queries = [
    "How do I fix 503-AUTH-TIMEOUT errors?",
    "What is SKU-2847-B?",
    "How can I extend authentication timeouts?"
]

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print('='*80)
    
    results = production_retriever.get_relevant_documents(query)
    
    for i, doc in enumerate(results, 1):
        score = doc.metadata.get('rerank_score', 0)
        print(f"\n{i}. [Score: {score:.3f}] {doc.metadata['doc_id']}")
        print(f"   {doc.page_content[:150]}...")
    
    # Calculate context statistics
    stats = calculate_context_stats(results)
    print(f"\nContext Stats: {stats['num_documents']} docs, "
          f"~{stats['total_tokens']} tokens, "
          f"avg score: {stats['avg_score']:.3f}")

## Next Steps

1. **Integrate with LLM**: Use the retrieved context with GPT-4 or Claude
2. **Tune Weights**: Experiment with BM25/Dense weights for your domain
3. **Add Your Data**: Replace sample documents with your knowledge base
4. **Measure Performance**: Track accuracy, latency, and cost metrics
5. **Deploy to Production**: See README.md for deployment options

---

**Read the full blog post**: `../blog-post-01-hybrid-search-reranking.md`